In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
NVIDIA GeForce RTX 5050 Laptop GPU
13.0


In [2]:
import pandas as pd
from pathlib import Path

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os
import datetime


load_dotenv()

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")


# 1. Charger le DataFrame filtré
df = pd.read_csv(
    "b_hashed_list.csv",
    usecols=[
        "title",
        # "ref",
        "status",
        "CELEX number",
        "Author",
        "Date of document",
        "link",
        # "Latest consolidated version",
        "hash_id"
    ]
)


#clean & prepare metadata
df = df.set_index("hash_id")
df["language"] = "english"
df["data source"] = "eur-lex"
df["date of migration"] = str(datetime.datetime.now().date())
df["Date of document"] = (
    df["Date of document"]
    .astype(str)
    .str.replace(r"[:;].*$", "", regex=True)
    .str.strip()
)
df.columns = df.columns.str.replace(":", "")
df.to_csv("b_hashed_list.csv")

In [1]:
import pandas as pd
df = pd.read_csv(
    "b_hashed_list.csv",
    index_col="hash_id",
    usecols=[
        "title",
        # "ref",
        "status",
        "CELEX number",
        "Author",
        "Date of document",
        "link",
        # "Latest consolidated version",
        "hash_id"
    ]
)


In [ ]:
from llama_index.core import SimpleDirectoryReader, StorageContext, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex
from qdrant_client import QdrantClient
import os
from pathlib import Path
import pandas as pd

# =========================
# CONFIG
# =========================
QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]


def add_file_metadata(path: str):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}

# =========================
# 1. DOCUMENTS
# =========================
documents = SimpleDirectoryReader(
    "xml_dir/",
    file_metadata=add_file_metadata
).load_data()

# documents = documents[:10]

print("len documents", len(documents))
# =========================
# 2. CHUNKING
# =========================
splitter = SentenceSplitter(
    chunk_size=800,
    chunk_overlap=100
)

nodes = splitter.get_nodes_from_documents(documents)
model_bge ="BAAI/bge-m3"
# =========================
# 3. EMBEDDINGS
# =========================
embed_model = HuggingFaceEmbedding(model_name=model_bge)
Settings.embed_model = embed_model

# =========================
# 4. QDRANT
# =========================
client = QdrantClient(path="./qdrant_local_bbai")

vector_store = QdrantVectorStore(
    client=client,
    collection_name="legal_BAAI_bge-m3"
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

# =========================
# 5. INDEXATION DIRECTE
# =========================
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context
)
index.storage_context.persist(persist_dir="index_storage_bbai")

print("Indexation complète OK.")


len documents 2634


2025-11-29 16:21:53,145 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3
c:\Users\alaa-\miniconda3\envs\tekno\lib\site-packages\llama_index\vector_stores\qdrant\base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(
c:\Users\alaa-\miniconda3\envs\tekno\lib\site-packages\qdrant_client\qdrant_client.py:2609: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  return self._client.upload_points(


Indexation complète OK.


In [ ]:
from llama_index.core import SimpleDirectoryReader, StorageContext, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex
from qdrant_client import QdrantClient
import os
from pathlib import Path
import pandas as pd

# =========================
# CONFIG
# =========================
QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]

# Choix du modèle d'embedding
# EMBED_MODEL_NAME = "BAAI/bge-m3"
# EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_MODEL_NAME = "BAAI/bge-m3"   # ← tu peux changer ici


def add_file_metadata(path: str):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}


# =========================
# 1. DOCUMENTS
# =========================
documents = SimpleDirectoryReader(
    "xml_dir/",
    file_metadata=add_file_metadata
).load_data()

print("len documents", len(documents))


# =========================
# 2. CHUNKING
# =========================
splitter = SentenceSplitter(
    chunk_size=800,
    chunk_overlap=100
)

nodes = splitter.get_nodes_from_documents(documents)


# =========================
# 3. EMBEDDINGS
# =========================
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.embed_model = embed_model


# =========================
# 4. QDRANT
# =========================
client = QdrantClient(path="./qdrant_local_bbai")

vector_store = QdrantVectorStore(
    client=client,
    collection_name="legal_" + EMBED_MODEL_NAME.replace("/", "_")
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)


# =========================
# 5. INDEXATION DIRECTE
# =========================
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context
)

index.storage_context.persist(persist_dir="index_storage_bbai")

print(f"Indexation complète OK avec le modèle : {EMBED_MODEL_NAME}")


In [14]:
#charger l'index plus tard
from llama_index.core import load_index_from_storage

vector_store = QdrantVectorStore(client=client, collection_name="legal_BAAI_bge-m3")

storage_context = StorageContext.from_defaults(
    persist_dir="index_storage_bbai",
    vector_store=vector_store
)

index = load_index_from_storage(storage_context, embed_model=embed_model)


2025-11-29 16:18:17,693 - INFO - Loading all indices.


In [17]:
q = "De quoi parle la Directive 2008/99/EC of the European Parliament? Explique moi les fondamentaux?"

In [18]:
# Création du query engine
query_engine = index.as_query_engine(
    similarity_top_k=5
)

# Faire une requête
response = query_engine.query(q)

print(response)


2025-11-29 19:55:24,026 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


La Directive 2008/99/EC du Parlement européen et du Conseil traite de la protection de l'environnement à travers le droit pénal. Les fondamentaux de cette directive incluent l'établissement de mesures en matière de droit pénal pour protéger l'environnement de manière plus efficace, en respectant les droits fondamentaux et en observant les principes reconnus par la Charte des droits fondamentaux de l'Union européenne.


In [19]:
query_engine = index.as_query_engine(
    similarity_top_k=20,
    rerank_top_k=5,
)
# Faire une requête
response = query_engine.query(q)

print(response)

2025-11-29 19:55:28,319 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


La Directive 2008/99/EC du Parlement européen et du Conseil du 19 novembre 2008 traite de la protection de l'environnement à travers le droit pénal. Les fondamentaux de cette directive incluent l'établissement de mesures relatives au droit pénal pour protéger l'environnement de manière plus efficace, en mettant en place des règles minimales et en permettant aux États membres d'adopter ou de maintenir des mesures plus strictes pour la protection de l'environnement, tout en respectant les droits fondamentaux et les principes reconnus par la Charte des droits fondamentaux de l'Union européenne.


In [ ]:
import os
import pandas as pd
from pathlib import Path

from llama_index.core import SimpleDirectoryReader, StorageContext, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.mistralai import MistralEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


# =========================
# CONFIGURATION
# =========================

# Variables d'environnement requises :
# export MISTRAL_API_KEY="xxx"
# export QDRANT_URL="http://localhost:6333"
# export QDRANT_API_KEY="xxx"

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY")
qdrant_url = os.environ.get("QDRANT_URL")
qdrant_api_key = os.environ.get("QDRANT_API_KEY")

# Charger ton dataframe contenant les métadonnées
df = pd.read_csv("metadata.csv", index_col=0)


# =========================
# 1. Ajout métadonnées → Document
# =========================

def add_file_metadata(path: str):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}


# =========================
# 2. Charger les documents
# =========================

documents = SimpleDirectoryReader(
    "texts/",
    file_metadata=add_file_metadata
).load_data()


# =========================
# 3. Découpage en chunks
# =========================

splitter = SentenceSplitter(
    chunk_size=700,
    chunk_overlap=100
)

nodes = splitter.get_nodes_from_documents(documents)


# =========================
# 4. Modèle d'embedding Mistral
# =========================

embed_model = MistralEmbedding(
    model="mistral-embed"
)

Settings.embed_model = embed_model


# =========================
# 5. Qdrant setup
# =========================

client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="eurlex"
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)


# =========================
# 6. Indexation
# =========================

from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context
)

print("✅ Indexation terminée avec Mistral embeddings")
